# From Variable Definitions to I-ADOPT Decomposition and Wikidata Linking



This notebook shows a **single, end-to-end run** of the I-ADOPT LLM service workflow:

1. **Phase 1 – Decomposition** of a variable definition into I-ADOPT-style components using
   - model: `qwen/qwen3-32b` very small model that can also be run locally using HPC infrastructre. F score close to 66%
   - model: `qwen/qwen3-235b-a22b-thinking-2507` A bigger model that gives better F score close to 70%
   - temperature: `0.5`
   - **5-shot** examples
   - **`matrix_extender`** prompt (using the revised matrix explanation).
2. **Phase 3 – Linking** these components to Wikidata entities using the **cross-encoder** approach:
   - model: `tomaarsen/Qwen3-Reranker-0.6B-seq-cls`
   - score threshold: **0.9** (only links when the score is high).

This notebook is derived from the benchmarking scripts but **does not** run experiments, grids, or metrics.
It only runs **one variable** through Phase 1 and Phase 3 in a transparent way.


Below are some packages that you need to install if you do not have them. Please use python 3.12.1 to save time.

In [2]:
# !pip install httpx
# !pip install python-dotenv
# !pip install openai
# !pip install sentence_transformers
# !pip install requests-cache
# !pip install torch


Please create a .env file and paste you openrounter.ai api key in front of OPENROUTER_API_KEY= before running below cell.

I have a restricted api key in the env file for now but it may have been expired or run out of limit by the time you are seeing this notebook. 

## 1. Setup


In [4]:
from __future__ import annotations

import json
import logging
import os
import pathlib
import re
import textwrap
import urllib.parse
from pprint import pprint
from typing import Any, Dict, List, Optional

import httpx
import requests
from dotenv import load_dotenv
from openai import APIStatusError, OpenAI, OpenAIError
from sentence_transformers import SentenceTransformer, CrossEncoder, util

load_dotenv()

# Initialize OpenAI client for OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

logging.basicConfig(level=logging.INFO)


## 2. Schema and Example Paths

Replace the placeholder paths below with the actual locations of your files:
- `Json_schema.json`
- 5-shot example JSONs for the `matrix_extender` prompt.


In [5]:
# ▪ Static configuration (paths are placeholders – replace them in your environment)
SCHEMA_PATH = pathlib.Path("data/Json_schema.json")
FIVE_SHOT_DIR = pathlib.Path("data/five_shot")

ONTO_KEYS = [
    "hasStatisticalModifier",
    "hasProperty",
    "hasObjectOfInterest",
    "hasMatrix",
    "hasContextObject",
    "hasConstraint",
]

# Load the JSON schema text
_SCHEMA_TEXT = SCHEMA_PATH.read_text(encoding="utf-8").strip()


## 3. Phase 1 – Prompt Construction (`matrix_extender`)

We now define the instructions and the prompt builder used in the Phase 1 experiments.
The important part here is the **`matrix_extender`** template, which embeds the revised matrix explanation.


In [6]:
# --------------------------------------------------------------------------- #
# ▪ Prompt helpers (from Phase 1 script)
# --------------------------------------------------------------------------- #

_SYSTEM_RULES = textwrap.dedent(
    """
    You are an ontology engineer.
    Your task is to output **one** JSON object that satisfies the
    JSON-Schema provided below.

    ▸ Copy *comment* verbatim from the user section.
    ▸ Do **NOT** introduce keys that are absent from the schema.
    ▸ Every value must respect the declared JSON type.
    ▸ Reply with the JSON object only — no markdown fences, no narration.
"""
).strip()

BASELINE_INSTRUCTIONS = _SYSTEM_RULES

REVISED_MATRIX_EXPLANATION = textwrap.dedent(
    """
    Additional guidance for this task

    Role summary:

    • hasProperty:
      The type of characteristic being observed
      (e.g. "distance", "mass flux", "temperature").

    • hasObjectOfInterest:
      The Entity whose Property is observed
      (e.g. "carbon", "organism", "habitat patch", "electron").

    • hasMatrix:
      An Entity or System that contains or surrounds the
      ObjectOfInterest. Examples: "soil", "ocean water",
      "organism", "solar wind", or a system such as
      "from vegetation to soil".

    • hasConstraint:
      Short phrases that limit the scope or state of the
      observation, such as "nearest neighbour", "dry",
      "at 15°C", "at non-limiting conditions",
      "due to ingestion".

    Deciding between hasMatrix and hasConstraint:

    • Use hasMatrix when you name an Entity or System
      that acts as the environment or container of the
      ObjectOfInterest.

    • Use hasConstraint for state or filter phrases that
      restrict a Property or Entity, even if they appear
      in the textual definition. For example:
        - "nearest neighbour"
        - "dry"
        - "at 15°C temperature"
        - "at non-limiting conditions"
        - "due to ingestion"
      These MUST NOT be placed in hasMatrix.

    Defining constraints (hasConstraint array):

    • Each constraint is an object with:
        - "label": a short phrase for the state or restriction.
        - "on": EXACTLY which Property or Entity this constraint applies to.

    • IMPORTANT RULE:
      The "on" value MUST be copied verbatim from one of
      the keys you already generated in this JSON:
        - the exact hasProperty string, OR
        - the exact entity label used in hasObjectOfInterest,
          hasMatrix, or hasContextObject.

      Do NOT invent new labels for "on".  
      Do NOT paraphrase.  
      Always reuse the exact string you already used elsewhere.

    • Examples:
        label = "nearest neighbour",    on = "distance"
        label = "dry",                  on = "soil"
        label = "due to ingestion",     on = "organism"
        label = "at 15°C temperature",  on = "temperature"

    Important exclusions:

    • Do NOT model units, instruments, methods, or
      geographical locations in this JSON. These should
      not appear in hasProperty, hasObjectOfInterest,
      hasMatrix, or hasConstraint.
"""
).strip()

PROMPT_TEMPLATES = {
    "baseline": BASELINE_INSTRUCTIONS,
    "matrix_extender": BASELINE_INSTRUCTIONS + "\n\n" + REVISED_MATRIX_EXPLANATION,
}

_EXAMPLE_HDR = "\n\n### Examples (valid against the same schema)\n"
_USER_HDR = "\n\n### Variable to decompose\n"
_EXPECTED = "\n\n### Expected output\n*(only the JSON object)*"


def build_prompt(comment: str, examples: List[Dict[str, Any]] | None, prompt_version: str) -> str:
    """Build the full prompt for a given variable comment using the chosen template."""
    examples = examples or []

    instructions = PROMPT_TEMPLATES.get(prompt_version, PROMPT_TEMPLATES["baseline"])

    ex_block = (
        _EXAMPLE_HDR + "\n\n".join(json.dumps(e, indent=2, ensure_ascii=False) for e in examples)
        if examples
        else ""
    )

    return (
        f"{instructions}\n\n"
        f"### JSON-Schema\n{_SCHEMA_TEXT}\n"
        f"{ex_block}"
        f"{_USER_HDR}comment: {comment}"
        f"{_EXPECTED}"
    )


## 4. Phase 1 – Few-Shot Examples

We load 5-shot examples from the `FIVE_SHOT_DIR`. Replace the directory path earlier
to point to the JSON files you used in your experiments.


In [7]:
def load_examples(n: int) -> List[Dict[str, Any]]:
    if n == 0:
        return []
    elif n == 5:
        folder = FIVE_SHOT_DIR
    else:
        raise ValueError("shot must be 5")
    paths = sorted(folder.glob("*.json"))
    return [json.load(open(p, encoding="utf-8")) for p in paths[:n]]

# Use 5-shot examples for the matrix_extender prompt
EXAMPLES_5SHOT = load_examples(5)


## 5. Phase 1 – LLM Invocation

We reuse the robust helpers from the Phase 1 script:
- `call_model` – handles retries and API glitches.
- `call_llm_loose` – extracts and normalises the JSON prediction.


In [8]:
# --------------------------------------------------------------------------- #
# ▪ LLM invocation helpers (Phase 1 only)
# --------------------------------------------------------------------------- #
_JSON_FENCE_RE = re.compile(r"```(?:json)?", re.MULTILINE)
_JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)


def call_model(model: str, prompt: str, temperature: float) -> str:
    """Robust API call with up to 3 retries."""
    for attempt in range(1, 4):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=temperature,
                messages=[{"role": "user", "content": prompt}],
                timeout=60,
            )
            text = resp.choices[0].message.content or ""

            # detect HTML or empty responses
            if text.strip().startswith("<!DOCTYPE html") or text.strip().startswith("<html"):
                logging.warning(f"{model}: HTML error response on attempt {attempt}")
                continue

            if not text.strip():
                logging.warning(f"{model}: empty response on attempt {attempt}")
                continue

            return text

        except APIStatusError as e:
            logging.warning(f"{model}: APIStatusError attempt {attempt} – {e.status_code} – {getattr(e, 'body', '')}")
        except (OpenAIError, httpx.HTTPError) as e:
            logging.warning(f"{model}: transport error attempt {attempt} – {e!r}")
        except Exception as e:
            logging.warning(f"{model}: unexpected error attempt {attempt} – {e!r}")

    logging.error(f"{model}: failed after 3 attempts")
    return ""


def call_llm_loose(model: str, prompt: str, comment: str, temperature: float) -> Dict[str, Any]:
    """
    Phase 1 helper:
    - Calls the model with retries.
    - Extracts the first JSON object from the response.
    - Ensures all ontology keys are present.
    """
    for attempt in range(1, 4):
        raw = call_model(model, prompt, temperature)

        if not raw.strip():
            logging.warning(f"{model}: empty output on JSON extraction attempt {attempt}")
            continue

        cleaned = _JSON_FENCE_RE.sub("", raw).strip()
        m = _JSON_BLOCK_RE.search(cleaned)
        if not m:
            logging.warning(f"{model}: no JSON block found on attempt {attempt}")
            continue

        try:
            data = json.loads(m.group(0))
        except Exception as e:
            logging.warning(f"{model}: JSON decode failure on attempt {attempt} – {e!r}")
            continue

        # success → post-process and return
        data["comment"] = comment
        for key in ONTO_KEYS:
            if key not in data:
                data[key] = [] if key == "hasConstraint" else ""
        return data

    logging.error(f"{model}: could not extract JSON after 3 attempts")
    return {}


## 6. Example Variable to Decompose

We will use the following variable:

```json
{
  "label": "Mass concentration of isobutylene in chloroform",
  "comment": "Mass concentration of isobutylene in chloroform.",
  "hasProperty": "mass concentration",
  "hasPropertyURI": "https://www.wikidata.org/wiki/Q589446",
  "hasMatrix": "chloroform",
  "hasMatrixURI": "https://www.wikidata.org/wiki/Q172275",
  "hasObjectOfInterest": "isobutylene",
  "hasObjectOfInterestURI": "https://www.wikidata.org/wiki/Q776976"
}
```

In this notebook, we only need the **label** and **comment** as input to the LLM.
The URIs above represent a reference solution you can use later for comparison.


In [15]:
variable_gt = {
    "label": "Mass concentration of isobutylene in chloroform",
    "comment": "Mass concentration of isobutylene in chloroform.",
    "hasProperty": "mass concentration",
    "hasPropertyURI": "https://www.wikidata.org/wiki/Q589446",
    "hasMatrix": "chloroform",
    "hasMatrixURI": "https://www.wikidata.org/wiki/Q172275",
    "hasObjectOfInterest": "isobutylene",
    "hasObjectOfInterestURI": "https://www.wikidata.org/wiki/Q776976",
}

variable_comment = variable_gt["comment"]
print(variable_comment)


Mass concentration of isobutylene in chloroform.


## 7. Run Phase 1 – Decomposition with `qwen/qwen3-32b`

We build a `matrix_extender` prompt with 5-shot examples and call the model once.


In [10]:
MODEL_PHASE1 = "qwen/qwen3-32b"
TEMPERATURE = 0.5
SHOT = 5
PROMPT_VERSION = "matrix_extender"

prompt = build_prompt(
    comment=variable_comment,
    examples=EXAMPLES_5SHOT,
    prompt_version=PROMPT_VERSION,
)

print("=== Phase 1 Prompt ===")
print(prompt + "\n...")

phase1_output = call_llm_loose(
    model=MODEL_PHASE1,
    prompt=prompt,
    comment=variable_comment,
    temperature=TEMPERATURE,
)

print("\n=== Phase 1 Output ===")
pprint(phase1_output)


=== Phase 1 Prompt ===
You are an ontology engineer.
Your task is to output **one** JSON object that satisfies the
JSON-Schema provided below.

▸ Copy *comment* verbatim from the user section.
▸ Do **NOT** introduce keys that are absent from the schema.
▸ Every value must respect the declared JSON type.
▸ Reply with the JSON object only — no markdown fences, no narration.

Additional guidance for this task

Role summary:

• hasProperty:
  The type of characteristic being observed
  (e.g. "distance", "mass flux", "temperature").

• hasObjectOfInterest:
  The Entity whose Property is observed
  (e.g. "carbon", "organism", "habitat patch", "electron").

• hasMatrix:
  An Entity or System that contains or surrounds the
  ObjectOfInterest. Examples: "soil", "ocean water",
  "organism", "solar wind", or a system such as
  "from vegetation to soil".

• hasConstraint:
  Short phrases that limit the scope or state of the
  observation, such as "nearest neighbour", "dry",
  "at 15°C", "at non-li

INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"



=== Phase 1 Output ===
{'comment': 'Mass concentration of isobutylene in chloroform.',
 'hasConstraint': [],
 'hasContextObject': '',
 'hasMatrix': 'chloroform',
 'hasObjectOfInterest': 'isobutylene',
 'hasProperty': 'mass concentration',
 'hasStatisticalModifier': '',
 'label': 'Mass concentration of isobutylene in chloroform'}


## 8. Phase 3 – Linking to Wikidata (Cross-Encoder Only)

We now reuse the Phase 3 linking logic from the benchmark script, but restrict
ourselves to the **cross-encoder** approach with a **0.9** score threshold.


In [12]:
# --------------------------------------------------------------------------- #
# ▪ Phase 3 – URI linking to Wikidata (from Phase 3 script)
# --------------------------------------------------------------------------- #

try:
    import requests_cache

    _CACHE_SESSION = requests_cache.CachedSession("wikidata_cache", backend="sqlite", expire_after=None)
    _REQUESTS = _CACHE_SESSION
except Exception:
    _REQUESTS = requests  # fallback without cache


def _qid_from_uri_or_text(s: Optional[str]) -> Optional[str]:
    if not s:
        return None
    m = re.search(r"(Q\d+)", s)
    return m.group(1) if m else None


def canonicalize_uri_for_compare(uri: Optional[str]) -> Optional[str]:
    """Make http/https equivalent and wiki/entity equivalent by canonicalizing to https://www.wikidata.org/wiki/Qxxxx."""
    if not uri:
        return None
    q = _qid_from_uri_or_text(uri)
    if q:
        return f"https://www.wikidata.org/wiki/{q}"
    # Fallback: normalize scheme & strip trailing slash
    u = uri.strip().replace("http://", "https://")
    return u[:-1] if u.endswith("/") else u


def _to_wiki_url(uri: Optional[str]) -> Optional[str]:
    """Convert any Q-id or wikidata URI to canonical https://www.wikidata.org/wiki/Qxxxx for storage in ...URI."""
    if not uri:
        return None
    q = _qid_from_uri_or_text(uri)
    return f"https://www.wikidata.org/wiki/{q}" if q else canonicalize_uri_for_compare(uri)


def format_queries(query, instruction=None):
    prefix = '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
    if instruction is None:
        instruction = "Given a web search query, retrieve relevant passages that answer the query"
    return f"{prefix}<Instruct>: {instruction}\n<Query>: {query}\n"


def format_document(document):
    suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
    return f"<Document>: {document}{suffix}"


# We only *use* the cross-encoder approach in this notebook.
# @functools.lru_cache(maxsize=2)
def load_crossencoder(model_name: str = "tomaarsen/Qwen3-Reranker-0.6B-seq-cls") -> CrossEncoder:
    return CrossEncoder(model_name)


In [13]:
def get_wikidata_entity(
    term: str,
    approach: str = "cross-encoder",
    context: str = "",
    model_name: str = "tomaarsen/Qwen3-Reranker-0.6B-seq-cls",
    threshold: float = 0.9,
) -> Optional[str]:
    """Return a Wikidata URI for *term* using the cross-encoder approach only in this notebook."""
    if not term:
        return None
    encoded = urllib.parse.quote_plus(term)
    headers = {"User-Agent": "IADOPT-Linker/1.0 (+notebook example)"}
    try:
        resp = _REQUESTS.get(
            f"https://www.wikidata.org/w/api.php?action=wbsearchentities&search={encoded}&language=en&format=json",
            headers=headers,
            timeout=20,
        )
        if resp.status_code != 200:
            logging.warning("Wikidata API HTTP %s for %r", resp.status_code, term)
            return None
        search = resp.json().get("search", [])

        if not search:
            return None

        # --- Cross-encoder re-ranking ---
        model = load_crossencoder(model_name)
        task = "Given a web search query, retrieve relevant passages that answer the query"
        queries = [f'Definition of "{term}" in context: "{context}"'] * len(search)
        documents = [
            f'label: "{search_entry.get("label","")}", description: "{search_entry.get("description","")}"'
            for search_entry in search
        ]
        pairs = [[format_queries(query, task), format_document(doc)] for query, doc in zip(queries, documents)]
        scores = model.predict(pairs)
        ranked = sorted(zip(search, scores), key=lambda x: x[1], reverse=True)

        best_s, best_score = ranked[0]
        logging.info(
            "Cross-encoder best candidate | term=%r | label=%r | score=%.4f | id=%s",
            term, best_s.get("label"), float(best_score), best_s.get("id")
        )

        if float(best_score) >= float(threshold):
            return _to_wiki_url(best_s["id"])
        return None

    except Exception as e:
        logging.warning("Wikidata API error for %r: %r", term, e)
        return None


def enrich_with_uris(
    pred: Dict[str, Any],
    approach: str = "cross-encoder",
    model_name: str = "tomaarsen/Qwen3-Reranker-0.6B-seq-cls",
    threshold: float = 0.9,
) -> Dict[str, Any]:
    """
    Return a copy of *pred* enriched with ...URI fields using Wikidata and a cross-encoder ranking.
    """
    out = json.loads(json.dumps(pred))  # deep copy

    def add_uri_field(container: Dict[str, Any], key: str, label_value: Any):
        if isinstance(label_value, str) and label_value.strip():
            uri = get_wikidata_entity(
                label_value,
                approach=approach,
                context=pred.get("label", ""),
                model_name=model_name,
                threshold=threshold,
            )
            if uri:
                container[f"{key}URI"] = uri

    # Top-level simple keys
    for p, key in [
        ("hasProperty", "hasProperty"),
        ("hasMatrix", "hasMatrix"),
        ("hasObjectOfInterest", "hasObjectOfInterest"),
        ("hasContextObject", "hasContextObject"),
    ]:
        if p in out and isinstance(out[p], str):
            add_uri_field(out, key, out[p])

    # Systems (nested) – kept for completeness, though our example uses simple strings
    for p in ["hasMatrix", "hasObjectOfInterest", "hasContextObject"]:
        val = out.get(p)
        if isinstance(val, dict):
            # Asymmetric
            if "AsymmetricSystem" in val:
                sys_lbl = val.get("AsymmetricSystem")
                src_lbl = val.get("hasSource")
                tgt_lbl = val.get("hasTarget")
                if sys_lbl:
                    uri = get_wikidata_entity(
                        sys_lbl,
                        approach=approach,
                        context=pred.get("label", ""),
                        model_name=model_name,
                        threshold=threshold,
                    )
                    if uri:
                        val["AsymmetricSystemURI"] = _to_wiki_url(uri)
                if src_lbl:
                    uri = get_wikidata_entity(
                        src_lbl,
                        approach=approach,
                        context=pred.get("label", ""),
                        model_name=model_name,
                        threshold=threshold,
                    )
                    if uri:
                        val["hasSourceURI"] = _to_wiki_url(uri)
                if tgt_lbl:
                    uri = get_wikidata_entity(
                        tgt_lbl,
                        approach=approach,
                        context=pred.get("label", ""),
                        model_name=model_name,
                        threshold=threshold,
                    )
                    if uri:
                        val["hasTargetURI"] = _to_wiki_url(uri)

            # Symmetric
            if "SymmetricSystem" in val:
                sys_lbl = val.get("SymmetricSystem")
                parts = val.get("hasPart", [])
                if sys_lbl:
                    uri = get_wikidata_entity(
                        sys_lbl,
                        approach=approach,
                        context=pred.get("label", ""),
                        model_name=model_name,
                        threshold=threshold,
                    )
                    if uri:
                        val["SymmetricSystemURI"] = _to_wiki_url(uri)
                if isinstance(parts, list) and parts:
                    part_uris: List[Optional[str]] = []
                    for part in parts:
                        if isinstance(part, str) and part.strip():
                            uri = get_wikidata_entity(
                                part,
                                approach=approach,
                                context=pred.get("label", ""),
                                model_name=model_name,
                                threshold=threshold,
                            )
                            part_uris.append(_to_wiki_url(uri) if uri else None)
                        else:
                            part_uris.append(None)
                    if any(part_uris):
                        val["hasPartURIs"] = part_uris
    return out


## 9. Run Phase 3 – Enrich with Wikidata URIs

We now call `enrich_with_uris` on the Phase 1 output. The cross-encoder will only
accept a Wikidata candidate if its score is at least **0.9**.


In [14]:
linked_output = enrich_with_uris(
    pred=phase1_output,
    approach="cross-encoder",
    model_name="tomaarsen/Qwen3-Reranker-0.6B-seq-cls",
    threshold=0.9,
)

print("=== Phase 3 Output (with URIs where available) ===")
pprint(linked_output)


INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: mps
Batches: 100%|██████████| 1/1 [00:02<00:00,  2.28s/it]
INFO:root:Cross-encoder best candidate | term='mass concentration' | label='mass concentration' | score=0.9998 | id=Q589446
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: mps
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.17it/s]
INFO:root:Cross-encoder best candidate | term='chloroform' | label='chloroform' | score=0.9903 | id=Q172275
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: mps
Batches: 100%|██████████| 1/1 [00:00<00:00,  3.64it/s]
INFO:root:Cross-encoder best candidate | term='isobutylene' | label='isobutylene' | score=0.9980 | id=Q776976


=== Phase 3 Output (with URIs where available) ===
{'comment': 'Mass concentration of isobutylene in chloroform.',
 'hasConstraint': [],
 'hasContextObject': '',
 'hasMatrix': 'chloroform',
 'hasMatrixURI': 'https://www.wikidata.org/wiki/Q172275',
 'hasObjectOfInterest': 'isobutylene',
 'hasObjectOfInterestURI': 'https://www.wikidata.org/wiki/Q776976',
 'hasProperty': 'mass concentration',
 'hasPropertyURI': 'https://www.wikidata.org/wiki/Q589446',
 'hasStatisticalModifier': '',
 'label': 'Mass concentration of isobutylene in chloroform'}


## 10. Convenience Wrapper

For reuse, you can wrap the whole pipeline into a single function.


In [16]:
def process_variable(comment: str) -> Dict[str, Any]:
    """Run Phase 1 (decomposition) and Phase 3 (Wikidata linking) for a single variable comment."""
    prompt = build_prompt(comment=comment, examples=EXAMPLES_5SHOT, prompt_version="matrix_extender")
    phase1 = call_llm_loose(model=MODEL_PHASE1, prompt=prompt, comment=comment, temperature=TEMPERATURE)
    linked = enrich_with_uris(
        pred=phase1,
        approach="cross-encoder",
        model_name="tomaarsen/Qwen3-Reranker-0.6B-seq-cls",
        threshold=0.9,
    )
    return linked

# Example: rerun for the same variable
linked_again = process_variable(variable_comment)
print("=== Linked output via wrapper ===")
pprint(linked_again)


INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: mps
Batches: 100%|██████████| 1/1 [00:02<00:00,  2.29s/it]
INFO:root:Cross-encoder best candidate | term='mass concentration' | label='mass concentration' | score=0.9998 | id=Q589446
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: mps
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.53it/s]
INFO:root:Cross-encoder best candidate | term='chloroform' | label='chloroform' | score=0.9903 | id=Q172275
INFO:sentence_transformers.cross_encoder.CrossEncoder:Use pytorch device: mps
Batches: 100%|██████████| 1/1 [00:02<00:00,  2.87s/it]
INFO:root:Cross-encoder best candidate | term='isobutylene' | label='isobutylene' | score=0.9980 | id=Q776976


=== Linked output via wrapper ===
{'comment': 'Mass concentration of isobutylene in chloroform.',
 'hasConstraint': [],
 'hasContextObject': '',
 'hasMatrix': 'chloroform',
 'hasMatrixURI': 'https://www.wikidata.org/wiki/Q172275',
 'hasObjectOfInterest': 'isobutylene',
 'hasObjectOfInterestURI': 'https://www.wikidata.org/wiki/Q776976',
 'hasProperty': 'mass concentration',
 'hasPropertyURI': 'https://www.wikidata.org/wiki/Q589446',
 'hasStatisticalModifier': '',
 'label': 'Mass concentration of isobutylene in chloroform'}


## 11. Conclusion

This notebook demonstrates a **single-pass, best-configuration** pipeline:

- **Phase 1** (decomposition):
  - `qwen/qwen3-32b`
  - temperature `0.5`
  - 5-shot examples
  - `matrix_extender` prompt.
- **Phase 3** (linking to Wikidata):
  - Cross-encoder model `tomaarsen/Qwen3-Reranker-0.6B-seq-cls`
  - Threshold `0.9` to ensure high-confidence links only.

You can now adapt:
- the paths to your own schema and example files,
- the variable definitions you want to decompose,
- or swap out model names if you extend your experiments.


## 12. Future Work and Project Roadmap

The workflow demonstrated in this notebook represents the **baseline version** of the I-ADOPT LLM Service.
It relies on:

* a **single prompt architecture** (`matrix_extender`)
* **five-shot examples**
* a **generic LLM-based decomposition**
* and a **cross-encoder-based Wikidata linking pipeline**

However, the larger project goes well beyond this initial prototype.
Below is an overview of the planned next steps and research directions.

---

## 🚀 12.1 Moving Beyond Simple Prompting: Decision Trees + Pattern Libraries

Until now, decomposition has relied on a **single prompt** guiding the model through the I-ADOPT structure.
This is effective but limited: it treats every variable using the same strategy and cannot explicitly encode prior domain knowledge.

We are currently developing a **Decision Tree + Pattern Library** approach that uses:

* **100 manually curated ground-truth variables**
* **expert-designed decomposition patterns**
* **variable-type decision trees**
* **retrieval-augmented pattern selection (RAG)**

The idea is:

1. A new variable is classified using a decision tree 
2. The system retrieves the **pattern** refered to in the decision tree.
3. The LLM uses this pattern **instead of a general prompt**.
4. The output becomes:

   * far more stable,
   * less sensitive to prompt variability,
   * significantly more accurate,
   * and much less prone to hallucination.

This will create a **multi-pattern RAG-guided decomposition system** instead of a single-prompt system.

---

## 🌐 12.2 Linking to Multiple Controlled Vocabularies

Currently, Phase 3 linking is limited to **Wikidata** using a cross-encoder reranker.
The next phase extends this to **multiple controlled vocabularies**, by:

1. Allowing the user to **select which vocabularies** they want to target and a recommended set of controlled vocabularies based on what domain is this variable from.
2. For each vocabulary, generating a set of candidate IRIs.
3. Applying the **same cross-encoder reranking strategy** to pick the best match.
4. Returning a combined and prioritized set of IRI links.

This generalizes the system into a **multivocabulary semantic linker**, making the output interoperable across different knowledge organization systems (KOS).

---

## 📄 12.3 Publication (Very Brief Mention)

A research article documenting this methodology—
**combining prompt-based decomposition and cross-encoder semantic linking**—
is currently being prepared for submission to the *Semantic Web – Interoperability, Usability, Applicability* journal.

It is planned for the upcoming **Special Issue on “Bridging Machine Learning and Knowledge Representation”**,
which focuses on integrating large language models and symbolic reasoning for robust, explainable, and scalable Semantic Web applications.

---

## 🛠️ 12.4 Towards a Full I-ADOPT LLM Service

Once the components mature, the final integrated system will support:

* Multiple LLMs (Qwen 32B, Qwen 235B Thinking, future open models)
* Pattern-aware variable decomposition
* Multi-KOS entity linking
* Confidence scores for each part of the decomposition
* Optional manual correction workflow with visualization helper
* Easy integration into variable cataloging tools
* publishable decomposed I-ADOPT variables that can be published using one submit key

---

## 🎯 Summary

This notebook demonstrates the **core pipeline**, but the ongoing research aims to transform it into a **neuro-symbolic decomposition system** that:

* combines LLMs with symbolic decision patterns,
* provides deterministic and reproducible decomposition,
* and integrates multiple controlled vocabularies using cross-encoder ranking.

